# 第 2 周练习：API 文档问答

一个小型的**API文档问答助手**。它使用轻量级检索步骤（工具）来提取相关的文档部分，然后通过引用回答您的问题。

**目标：** 展示工具使用+检索+结构化响应。

In [1]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 进口
import os
import re
import json
from dotenv import load_dotenv
from openai import OpenAI



In [2]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 加载环境变量（.env）
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key found. Please add OPENAI_API_KEY to your .env file.')
elif api_key.strip() != api_key:
    print('API key has leading/trailing whitespace. Please remove it.')
else:
    print('API key looks good!')

openai = OpenAI(
    api_key=api_key,
    base_url='https://openrouter.ai/api/v1',
)



API key looks good!


In [3]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 选型
MODEL = 'gpt-4.1-mini'



In [12]:
# 示例 API 文档（替换为您自己的文档）
API_DOCS = '''
# Acme 支付 API

# 验证
Use a bearer token in the Authorization header. Tokens expire after 24 hours.

# 创建费用（POST /v1/费用）
Required fields: amount (integer, cents), currency (string), source (string token).
Optional: description, metadata.
Returns: charge_id, status, created_at.

# 退款费用（POST /v1/charges/{charge_id}/refunds）
Required fields: amount (integer, cents).
Optional: reason (string).
Returns: refund_id, status.

# 列出费用（GET /v1/charges）
Query params: status, limit, starting_after.
Returns a paginated list of charges.

# 网络钩子
We send events for charge.succeeded, charge.failed, refund.created.
Retry policy: 3 attempts over 24 hours.
'''



In [13]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 按标题简单的文档分块
def chunk_docs(text: str):
    chunks = []
    current = []
    for line in text.splitlines():
        if line.startswith('#'):  # new section
            if current:
                chunks.append('\n'.join(current).strip())
                current = []
        current.append(line)
    if current:
        chunks.append('\n'.join(current).strip())
    return [c for c in chunks if c]

DOC_CHUNKS = chunk_docs(API_DOCS)



In [14]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 检索工具：按关键词重叠返回top-k部分
def search_docs(query: str, k: int = 3):
    q = re.findall(r'\w+', query.lower())
    if not q:
        return []
    scores = []
    for chunk in DOC_CHUNKS:
        text = chunk.lower()
        score = sum(text.count(word) for word in q)
        scores.append((score, chunk))
    scores.sort(key=lambda x: x[0], reverse=True)
    return [c for s, c in scores if s > 0][:k]



In [15]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 函数调用的工具定义
TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'search_docs',
            'description': 'Search the API docs and return relevant sections.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string'},
                    'k': {'type': 'integer', 'default': 3},
                },
                'required': ['query']
            }
        }
    }
]



In [16]:
SYSTEM_PROMPT = '''
You are an API documentation assistant.
When needed, call the search_docs tool to retrieve relevant sections.
You MUST respond using these exact headings (with ##):
# 简答
# 细节
# 引文
Under Citations, use bullet points and quote the section titles used.
If the docs don't mention it, say so clearly.
If you fail to use the exact headings, the answer is invalid.
Example format:
# 简答
...
# 细节
...
# 引文
- "Section Title"
'''



In [17]:
def ask(question: str):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice='auto',
        max_tokens=800,
    )

    msg = response.choices[0].message
    if msg.tool_calls:
        # 处理工具调用
        tool_outputs = []
        for call in msg.tool_calls:
            if call.function.name == 'search_docs':
                args = json.loads(call.function.arguments) if hasattr(call.function, 'arguments') else {}
                query = args.get('query', question)
                k = args.get('k', 3)
                results = search_docs(query, k=k)
                tool_outputs.append({
                    'tool_call_id': call.id,
                    'role': 'tool',
                    'name': 'search_docs',
                    'content': '\n\n'.join(results) if results else 'No relevant sections found.'
                })

        # 将工具输出发送回模型
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages + [msg] + tool_outputs,
            max_tokens=800,
        )
        return response.choices[0].message.content

    return msg.content



In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 示例问题（一次运行一个）
print(ask('How do I refund a charge?'))
print(ask('What webhook events exist, and how often are retries?'))



## Short Answer
You refund a charge by making a POST request to the endpoint `/v1/charges/{charge_id}/refunds` with the required amount to refund and optional reason.

## Details
To refund a charge, use the endpoint `POST /v1/charges/{charge_id}/refunds`. You need to specify the amount to refund in cents as a required field. Optionally, you can provide a reason for the refund. The response will include the refund ID and status of the refund.

## Citations
- "Refund Charge (POST /v1/charges/{charge_id}/refunds)"
